# SAM 3 server for CVAT (Google Colab)

Runs **Segment Anything 3** on a free Colab GPU and exposes it to your local CVAT
through a Cloudflare tunnel.

**Runtime**: use `Runtime > Change runtime type > T4 GPU`.

After the last cell prints `COLAB_FUNCTION_URL`, on your PC run:

```
powershell -File sam3\colab\update-colab-url.ps1 -Url <COLAB_FUNCTION_URL>
```

then restart CVAT's server once:

```
docker restart cvat_server
```

Open a CVAT job and pick **Segment Anything 3 (Colab GPU)** in AI Tools > Interactors.
Keep the Colab tab open while annotating. The URL changes on every new session -
just re-run the last cell and `update-colab-url.ps1` afterwards.

## 1. Install dependencies and download the checkpoint

In [2]:
%pip install -q "numpy>=2.0" flask pillow requests iopath pycocotools psutil einops
%pip install -q --no-deps "git+https://github.com/facebookresearch/sam3.git"
!curl -L --fail -sS -o /content/sam3.pt https://huggingface.co/1038lab/sam3/resolve/main/sam3.pt
!ls -lh /content/sam3.pt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
-rw-r--r-- 1 root root 3.3G Nov 12 05:19 /content/sam3.pt


## 2. Load SAM 3 (with the CPU/CUDA compatibility patches)

In [3]:
# Install missing dependencies required by sam3 tokenizer
!pip install -q ftfy regex

import torch
from sam3.model_builder import build_sam3_image_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = "/content/sam3.pt"

model = build_sam3_image_model(
    device=DEVICE,
    checkpoint_path=CKPT,
    load_from_HF=False,
    enable_inst_interactivity=True,
)
# The inst-interactivity tracker is built without a backbone on purpose;
# it shares the main model's VL backbone (as in the official SAM 3 notebooks).
model.inst_interactive_predictor.model.backbone = model.backbone
# Checkpoint weights are bfloat16; cast everything to float32.
model = model.float()

# sam3.perflib.addmm_act hard-casts to bfloat16 (a CUDA-only optimization)
# and breaks fp32 inference in the ViT blocks. Use the plain fc1+GELU path.
import sam3.model.vitdet as _vitdet
def _plain_addmm_act(activation, linear, mat1):
    # activation may be a class (type(self.act)) or an instance
    act = activation() if isinstance(activation, type) else activation
    return act(linear(mat1))
_vitdet.addmm_act = _plain_addmm_act

predictor = model.inst_interactive_predictor
print(f"SAM 3 loaded on {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

/usr/local/lib/python3.13/dist-packages/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.laye

SAM 3 loaded on cuda (Tesla T4)


## 3. Define the Nuclio-compatible HTTP server

In [8]:
import base64, hashlib, io, json, threading
import numpy as np
from PIL import Image
from flask import Flask, jsonify, request

FUNCTION_ID = "pth-facebookresearch-sam3-colab"


def mask_to_rle(mask):
    """Convert a binary mask to CVAT's compressed RLE with the bbox suffix."""
    height, width = mask.shape
    pixels = (np.asarray(mask).reshape(-1) != 0).astype(np.uint8)
    if pixels.size == 0:
        return []
    changes = np.flatnonzero(pixels[1:] != pixels[:-1]) + 1
    rle = np.diff(np.concatenate(([0], changes, [pixels.size]))).tolist()
    if pixels[0] == 1:
        rle.insert(0, 0)
    rle.extend([0, 0, width - 1, height - 1])
    return rle


class Sam3Handler:
    def __init__(self):
        self.predictor = predictor
        self.latest_image_hash = None

    def set_image(self, image):
        image_hash = hashlib.sha1(image.tobytes()).digest()
        if image_hash != self.latest_image_hash:
            self.predictor.set_image(np.asarray(image))
            self.latest_image_hash = image_hash

    @staticmethod
    def _parse_box(obj_bbox):
        if not obj_bbox:
            return None
        if len(obj_bbox) == 2 and isinstance(obj_bbox[0], (list, tuple)):
            (x1, y1), (x2, y2) = obj_bbox
        else:
            x1, y1, x2, y2 = obj_bbox
        return [float(x1), float(y1), float(x2), float(y2)]

    def handle(self, image, pos_points, neg_points, obj_bbox):
        self.set_image(image)

        point_coords = [[float(x), float(y)] for x, y in [*pos_points, *neg_points]]
        point_labels = [1] * len(pos_points) + [0] * len(neg_points)
        box = self._parse_box(obj_bbox)

        if not point_coords and box is None:
            return None

        masks, ious, _ = self.predictor.predict(
            point_coords=np.asarray(point_coords, dtype=np.float32) if point_coords else None,
            point_labels=np.asarray(point_labels, dtype=np.int32) if point_labels else None,
            box=np.asarray(box, dtype=np.float32) if box is not None else None,
            multimask_output=len(point_coords) <= 1 and box is None,
        )
        best_mask = masks[int(np.argmax(ious))]
        return mask_to_rle(best_mask.astype(np.uint8))


sam3 = Sam3Handler()
sam3_lock = threading.Lock()

FUNCTION_ENTRY = {
    "metadata": {
        "name": FUNCTION_ID,
        "annotations": {
            "name": "Segment Anything 3 (Colab GPU)",
            "version": "2",
            "type": "interactor",
            "spec": "[]",
            "min_pos_points": "0",
            "min_neg_points": "0",
            "startswith_box_optional": "true",
            "help_message": (
                "The interactor allows to get a mask of an object using positive "
                "and negative points and/or a bounding box. Runs on a Google Colab GPU."
            ),
        },
    },
    "spec": {"description": "Interactive object segmentation with SAM 3 (Google Colab GPU)"},
    "status": {"httpPort": 8000},
}

app = Flask(__name__)


@app.get("/api/functions")
def list_functions():
    return jsonify({FUNCTION_ID: FUNCTION_ENTRY})


@app.get(f"/api/functions/{FUNCTION_ID}")
def get_function():
    return jsonify(FUNCTION_ENTRY)


@app.post("/api/function_invocations")
def invoke():
    if request.headers.get("x-nuclio-function-name", FUNCTION_ID) != FUNCTION_ID:
        return jsonify({"error": "unknown function"}), 404

    data = request.get_json(force=True)
    try:
        image = Image.open(io.BytesIO(base64.b64decode(data["image"]))).convert("RGB")

        with sam3_lock:
            points = sam3.handle(
                image,
                data.get("pos_points") or [],
                data.get("neg_points") or [],
                data.get("obj_bbox"),
            )
    except Exception as exc:
        import traceback
        traceback.print_exc()  # visible in the Colab cell output
        return jsonify({"error": f"SAM3 Colab server error: {exc}"}), 500

    shapes = []
    if points:
        shapes.append({
            "points": points,
            "group": 0,
            "source": "semi-auto",
            "attributes": [],
            "occluded": False,
            "rotation": 0,
            "type": "mask",
        })
    return jsonify({"shapes": shapes})

print("Server defined.")

Server defined.


## 4. Start the server and open the tunnel

In [9]:
import werkzeug.serving

# Restart cleanly if this cell is re-run (otherwise an old server keeps
# holding port 8000 with stale model state and returns 500s).
if "http_server" in globals():
    try:
        http_server.shutdown()
    except Exception as exc:
        print("old server shutdown:", exc)

http_server = werkzeug.serving.make_server("0.0.0.0", 8000, app, threaded=True)
threading.Thread(target=http_server.serve_forever, daemon=True).start()
print("Flask server started on port 8000")

 * Serving Flask app '__main__'
Flask server started on port 8000
 * Debug mode: off


Address already in use
Port 8000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [10]:
import re, subprocess, time

if "cloudflared" not in globals():
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

COLAB_FUNCTION_URL = None
start = time.time()
error_logs = []

for line in tunnel.stdout:
    # Log potential errors we find
    if "failed" in line.lower() or "error" in line.lower() or "429" in line:
        error_logs.append(line.strip())

    for match in re.finditer(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line):
        candidate = match.group(0)
        if not candidate.startswith("https://api."):
            COLAB_FUNCTION_URL = candidate
            break
    if COLAB_FUNCTION_URL:
        break
    if time.time() - start > 60:
        break

if COLAB_FUNCTION_URL:
    print("Copy this URL to your PC, then run update-colab-url.ps1:")
    print()
    print(f"    COLAB_FUNCTION_URL = {COLAB_FUNCTION_URL}")
    print()
    print("powershell -File sam3\\colab\\update-colab-url.ps1 -Url " + COLAB_FUNCTION_URL)
    print("docker restart cvat_server")
else:
    print("[-] Failed to establish a Cloudflare tunnel.")
    if any("429" in l for l in error_logs):
        print("\n[!] Reason: Cloudflare is currently rate-limiting quick tunnel creation (HTTP 429).")
        print("    Please wait 1-2 minutes and run this cell again to retry.")
    elif error_logs:
        print("\nLast logs:")
        for log in error_logs[-5:]:
            print(f"  {log}")

/content/cloudflared: Text file busy
Copy this URL to your PC, then run update-colab-url.ps1:

    COLAB_FUNCTION_URL = https://guru-fragrance-templates-prospective.trycloudflare.com

powershell -File sam3\colab\update-colab-url.ps1 -Url https://guru-fragrance-templates-prospective.trycloudflare.com
docker restart cvat_server


## 5. (Optional) Smoke test from Colab itself

In [7]:
import time, urllib.request

img = (np.random.rand(128, 128, 3) * 255).astype("uint8")
import PIL.Image
PIL.Image.fromarray(img).save("/content/test.jpg", quality=90)
with open("/content/test.jpg", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

payload = json.dumps({
    "image": img_b64,
    "pos_points": [[64, 64]],
    "neg_points": [],
    "obj_bbox": None,
}).encode()

req = urllib.request.Request(
    "http://localhost:8000/api/function_invocations",
    data=payload,
    headers={"Content-Type": "application/json"},
)
start = time.time()
reply = urllib.request.urlopen(req, timeout=120)
result = json.loads(reply.read())
print(f"elapsed_sec={time.time() - start:.1f} shapes={len(result['shapes'])}")

/usr/local/lib/python3.13/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()
ERROR:__main__:Exception on /api/function_invocations [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "/usr/local/lib/python3.13/dist-packages/flask/app.

HTTPError: HTTP Error 500: INTERNAL SERVER ERROR